# Synthetic stimulus generation

Run from the repository root. Creates 300 clips with target-component amplitude differences. This maintenance version fixes the NumPy seed for future runs; it does not recreate the original unseeded waveforms. Existing WAVs are never overwritten.

In [ ]:
### SIM
import numpy as np
from pathlib import Path
import soundfile as sf

output_dir = Path('sim_data')
if not Path('cfg/SincNet_Sim.cfg').is_file():
    raise RuntimeError('Run this notebook from the repository root.')
output_dir.mkdir(parents=True, exist_ok=True)
if any(output_dir.glob('*.wav')):
    raise FileExistsError('sim_data already contains WAVs; keep them or archive them before regenerating.')
np.random.seed(1234)  # Added during repository cleanup; original synthesis was unseeded.

def rampdamp(signal, fs, ramp_dur):
    ramp_len = int(ramp_dur * fs)
    ramp = np.linspace(0, 1, ramp_len)
    signal[:ramp_len] *= ramp
    signal[-ramp_len:] *= ramp[::-1]
    return signal

# Parameters
ntrials = 150
fs = 44100
dur = 0.5
t = np.arange(0, dur, 1/fs)

sigbearingf = [500, 1000, 1500, 2000, 2500]
sigbearingmean = 53  # with signal
nosigbearingmean = 50  # without signal
nosigbaseamps = 10  # never signal

freqs = np.arange(300, 2800, 20)
ntones = len(freqs)
nosigbase = nosigbaseamps * np.ones_like(freqs)
sigbase = nosigbase.copy()

idxsig = []
for f in sigbearingf:
    idx = np.where(freqs == f)[0][0]
    idxsig.append(idx)
    nosigbase[idx] = nosigbearingmean
    sigbase[idx] = sigbearingmean

sphase = 2 * np.pi * np.random.randn(ntrials, ntones)
nsphase = 2 * np.pi * np.random.randn(ntrials, ntones)

sig = []
noisig = []

for kk in range(ntrials):
    temps = np.zeros((ntones, len(t)))
    tempns = np.zeros((ntones, len(t)))
    sigamps = np.zeros(ntones)
    nosigamps = np.zeros(ntones)
    
    for jj in range(ntones):
        sigamps[jj] = sigbase[jj] + 2 * np.random.randn()
        nosigamps[jj] = nosigbase[jj] + 2 * np.random.randn()
        temps[jj, :] = sigamps[jj] * np.cos(2 * np.pi * freqs[jj] * t + sphase[kk, jj])
        tempns[jj, :] = nosigamps[jj] * np.cos(2 * np.pi * freqs[jj] * t + nsphase[kk, jj])
    
    temp = np.sum(temps, axis=0) / 300
    sig_kk = rampdamp(temp, fs, 0.005)
    sig.append(sig_kk)
    
    temp = np.sum(tempns, axis=0) / 300
    nosig_kk = rampdamp(temp, fs, 0.005)
    noisig.append(nosig_kk)
    
    # Save signals
    sig_fname = f'sim_data/sig{kk + 1}.wav'
    nosig_fname = f'sim_data/nosig{kk + 1}.wav'
    sf.write(sig_fname, sig_kk, fs)
    sf.write(nosig_fname, nosig_kk, fs)

sig = np.array(sig)
noisig = np.array(noisig)
